# Homework 3 - Classification (Lead Scoring 2026)

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## Data preparation

In [2]:
df = pd.read_csv("../../data/course_lead_scoring_2026.csv")

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["number_of_courses_viewed", "annual_income", "interaction_count", "lead_score"]
target = "converted"

df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [3]:
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum().sum()

np.int64(0)

## Question 1. Mode of `industry`

In [4]:
print(df["industry"].value_counts())
df["industry"].mode()[0]

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


'technology'

## Question 2. Correlation matrix

In [5]:
corr = df[numerical].corr()
corr

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.161300,0.721609,0.757204
annual_income,0.161300,1.000000,0.122842,0.229496
interaction_count,0.721609,0.122842,1.000000,0.915746
lead_score,0.757204,0.229496,0.915746,1.000000


In [6]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for a, b in pairs:
    print(f"{a} / {b}: {corr.loc[a, b]:.4f}")

interaction_count / lead_score: 0.9157
number_of_courses_viewed / lead_score: 0.7572
number_of_courses_viewed / interaction_count: 0.7216
annual_income / interaction_count: 0.1228


## Split the data

In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train.pop(target).values
y_val = df_val.pop(target).values
y_test = df_test.pop(target).values

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Question 3. Mutual information (training set only)

In [8]:
mi = {col: round(mutual_info_score(df_train[col], y_train), 2) for col in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Question 4. Logistic regression

In [9]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))


all_features = categorical + numerical
base_accuracy = train_and_score(all_features)
round(base_accuracy, 2)

0.65

## Question 5. Feature elimination

In [10]:
diffs = {}
for feature in all_features:
    features = [f for f in all_features if f != feature]
    diffs[feature] = base_accuracy - train_and_score(features)

print(pd.Series(diffs).sort_values())

candidates = ["lead_source", "number_of_courses_viewed", "interaction_count"]
min(candidates, key=lambda f: diffs[f])

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64


'number_of_courses_viewed'

## Question 6. Regularized logistic regression

In [11]:
scores = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    scores[C] = round(train_and_score(all_features, C=C), 3)
    print(C, scores[C])

best = max(scores.values())
min(C for C, s in scores.items() if s == best)

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


0.001